# RAG 고도화 전략 개요
RAG 고도화는 관련 문서를 가져오는 Retrieval 파트와 응답을 생성하는 Generation 파트 모두에서 달성 가능하다. 
## Retrieval 
### 청킹 전략(전처리)
- 문서를 효과적으로 분할하는 다양한 기법을 활용하여 검색에 최저고하된 형태로 변환, 검색 품질과 처리 속도를 향상한다. 
### 질의 변형 
- 질문의 구체화, 가상의 답변 활용 등의 기법을 통해 사용자의 원래 질문을 검색에 효율적인 형태로 재구성한다. 
### 검색 알고리즘 최적화 
- 의미 기반의 벡터 검색과 키워드 기반 검색, 하이브리드 검색 등의 방식을 활용하여 관련성이 높은 문서를 더 효율적으로 검색한다. 
### 문서 후처리(리랭킹)
- 1차 검색결과와 질문의 연관성을 재평가 하여 질문과 관련없는 문서 제거, 이를 통해 검색된 정보의 품질을 높이고 궁극적으로 더 신뢰할 수 있는 최종 프롬프트 제공한다. 
## Generation 
### Self-RAG 
- 모델이 스스로 추가 검색 필요성을 판단하고 부족한 정보를 찾아 보완하고 이를 바탕으로 더 깊이 있고 정확한 응답을 제공하여 사용자 경험을 개선한다. 
### 모델 파인 튜닝 
- 모델을 학습 시켜 다양한 상황에 대한 대응 능력을 강화하고 이를 통해 검색된 정보를 더윽 효율적으로 활용하여 맥락에 맞는 적절한 응답 생성이 가능하다. 

## 1. 청킹 
- 문서 전처리 단계는 RAG 시스템에서 실제 사용될 문서를 가공, 검색과 생성에 최적화된 형태로 변환하는 과정 
- 효과적인 전처리는 검색 정확도를 높이고 관련성 있는 정보를 더 잘 추출 가능하며 생성 모델의 응답 품질 향상 
- 문서 전처리 과정에서 핵심적인 단계가 청킹으로 문서 분할 단계이다. 
- 문서 분할은 긴 문서를 더 작고 관리하기 쉬운 단위로 나누는 과정을 의미한다. 
- 가장 기본적인 문서 분할 방식은 문자수 기반 분할이며 구현이 간단하고 빠르다. 
- 문자수 기반 분할은 문장, 단락의 의미적 구조를 고려하지 않기 때문에 중요한 정보가 분할되어서 검색시 누락되거나 문맥의 왜곡의 단점이 있다. 

### 부모-자식 분할 
- 부모 자식 분할은 문서를 계층적으로 분할하여 원본 문서를 큰 단위의 부모 문서로 나누고 이를 다시 작은 단위의 자식 문서로 세분화한다. 
- 원본 문서 -> 부모 문서 -> 자식 문서 3단계의 구조를 형성한다. 
- 부모-자식 분할 방식은 문서의 저장과 검색에서 이원화된 접근법을 채택한다. 
- 문서의 계층 구조를 유지하면서 효율적인 검색을 위해 자식 문서는 벡터 데이터베이스에 임베딩하여 저장하고 부모 문서는 별도의 저장소에 원본 형태로 보관한다. 
- 검색 시에는 자식 문서를 기반으로 유사성 검색을 수행하지만 반환되는 문서는 해당 자식 문서가 속한 부모 문서이다. 

In [15]:
from langchain_community.document_loaders import TextLoader 

loaders = [
    TextLoader("./data/How_to_invest_money.txt")
]

docs = []

for loader in loaders:
    docs.extend(loader.load())

In [16]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_community.vectorstores import Chroma 
from langchain_openai import OpenAIEmbeddings 
from langchain_text_splitters import RecursiveCharacterTextSplitter 


In [17]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)


In [18]:
import chromadb

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    tenant="default_tenant",
    database="default_database",
)

print(client.heartbeat())

1782706436945077760


In [19]:
embedding_function = OpenAIEmbeddings(
    model="bge-m3",
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False,
)

try:
    client.delete_collection("real_estate")
except Exception:
    pass 



In [21]:

vectorstore = Chroma(
    collection_name="split_parents", 
    embedding_function=embedding_function,
    client=client 
)

/var/folders/kc/qm9ykcl12cv910wgvjsx9hs40000gn/T/ipykernel_52108/3665860104.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [22]:
store = InMemoryStore()

In [ ]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore, 
    docstore =store, 
    child_splitter=child_splitter, 
    parent_splitter=parent_splitter,
)

retriever.add_documents(docs)

In [ ]:
print(f"Number of parent docuemnts : {len(list(store.yield_keys()))}")

Number of parent docuemnts : 219


In [28]:
query = "Whar are the types of investments?"

retrieved_docs = retriever.invoke(query)

print(f"retrieved_docs length {len(retrieved_docs)}")
print(f"Parent Document: {retrieved_docs[0].page_content}")

retrieved_docs length 4
Parent Document: There are five chief points to be considered in the selection of all
forms of investment. These are: (1) safety of principal and interest;
(2) rate of income; (3) convertibility into cash; (4) prospect of
appreciation in intrinsic value; (5) stability of market price.

Keeping these five general factors in mind, the present chapter will
discuss real-estate mortgages as a form of investment, both as adapted
to the requirements of private funds and of a business surplus.


In [29]:
query = "What are types of investments?"

sub_docs = vectorstore.similarity_search(query)
print(f"Child Document : {sub_docs[0].page_content}")

Child Document : forms of investment. These are: (1) safety of principal and interest;
(2) rate of income; (3) convertibility into cash; (4) prospect of
appreciation in intrinsic value; (5) stability of market price.
